# 🎥 Universal Video LoRA Trainer (Wan 2.1 & Wan 2.2)
Hệ thống huấn luyện LoRA Video chuyên sâu trên Google Colab hỗ trợ **Text-to-Video** và **Image-to-Video** với **Wan 2.1** và **Wan 2.2**.

---
### 📚 Hướng Dẫn Chuẩn Bị Video Dataset:
- Chuẩn bị 10 - 50 video clips ngắn (3 - 10 giây mỗi clip), định dạng `.mp4`.
- Tỉ lệ khuyến nghị: 16:9 (`720,1280`) hoặc 9:16 (`1280,720`).
- Đặt lượng frame mục tiêu (`Target_Frames`): 25, 33, 49 hoặc 81 frames.

### ☕ Bước 1: Khởi tạo Môi trường & Kiểm tra GPU

In [ ]:
# @title ⚙️ 1. Cài đặt Môi trường
import os
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

!pip install -q toml pyyaml python-dotenv bitsandbytes optimum-quanto google-genai openai accelerate safetensors huggingface_hub tqdm pillow av opencv-python-headless

if not os.path.exists('/content/TranningLoras'):
    !git clone https://github.com/nguyenducvuongg/TranningLoras.git /content/TranningLoras
else:
    !git -C /content/TranningLoras pull

%cd /content/TranningLoras
!pip install -q -e .

from lora_trainer.engine.hardware import detect_hardware_environment
hw = detect_hardware_environment()
print(f"🚀 GPU: {hw['gpu_name']} | VRAM: {hw['vram_gb']} GB")

### 📂 Bước 2: Dữ liệu Video, Frame Slicing & AI Captioning

In [ ]:
# @title 📂 2. Xử lý Dữ liệu Video & Cắt Frame
Video_Folders = "/content/drive/MyDrive/LoRA_Video_Data" # @param {type:'string'}

# @markdown 🎞️ **Cấu hình Trích xuất Khung hình (Frame Extraction)**
Frame_Extraction = "chunk" # @param ["chunk", "slide", "uniform", "head", "full"]
Target_Frames = "25" # @param {type:'string'}
Frame_Stride = 1 # @param {type:'integer'}
Frame_Sample = 1 # @param {type:'integer'}
Max_Frames = 33 # @param {type:'integer'}

# @markdown 🤖 **Tự động gán nhãn Video bằng Gemini API (Đọc video trực tiếp)**
Auto_Caption_Video = False # @param {type:'boolean'}
Gemini_Model = "Gemini-2.5-Flash" # @param ["Gemini-2.5-Flash", "Gemini-2.5-Pro", "Gemini-2.5-Flash-Lite", "Gemini-2.0-Flash", "Gemini-1.5-Pro", "Gemini-1.5-Flash"]
Gemini_API_Key = "" # @param {type:'string'}
Custom_Tag = "" # @param {type:'string'}

from lora_trainer.data.cleaner import clean_directory, get_supported_videos
from lora_trainer.caption.gemini_captioner import batch_caption_gemini
from lora_trainer.data.tag_processor import process_dir_tags

for v_dir in [d.strip() for d in Video_Folders.split(",") if d.strip()]:
    clean_directory(v_dir)
    if Auto_Caption_Video:
        batch_caption_gemini(v_dir, api_key=Gemini_API_Key, model_alias=Gemini_Model, is_video_folder=True)
    if Custom_Tag:
        process_dir_tags(v_dir, Custom_Tag)
    vids = get_supported_videos(v_dir)
    print(f"📹 Thư mục {v_dir}: {len(vids)} video.")

### 🚀 Bước 3: Cấu hình Wan & Bắt đầu Huấn luyện

In [ ]:
# @title 🛠️ 3. Cấu hình Wan 2.1 / Wan 2.2 & Bắt đầu Huấn luyện
Model_Type = "Wan2.2-T2V-14B" # @param ["Wan2.1-T2V-14B", "Wan2.1-I2V-14B-720P", "Wan2.1-I2V-14B-480P", "Wan2.1-T2V-1.3B", "Wan2.2-T2V-14B", "Wan2.2-I2V-14B"]

Output_Directory = "/content/drive/MyDrive/LoRA_Video_Outputs" # @param {type:'string'}
LoRA_Name = "my_video_lora" # @param {type:'string'}

Resolution = "720,1280" # @param {type:'string'}
Learning_Rate = 1e-4 # @param {type:'number'}
Num_Repeats = 10 # @param {type:'integer'}
Max_Train_Epochs = 5 # @param {type:'integer'}
Save_Every_N_Epochs = 1 # @param {type:'integer'}

# @markdown 💡 **Timestep Sampling & Wan 2.2 Boundary**
Timestep_Sampling = "shift" # @param ["shift", "sigma", "uniform", "sigmoid", "logsnr"]
Timestep_Boundary = 875 # @param {"type":"slider","min":0,"max":1000,"step":5}
Sample_Prompt = "" # @param {type:'string'}
Sample_Every_N_Steps = 200 # @param {type:'integer'}

Auto_Disconnect = False # @param {type:'boolean'}

import os
from lora_trainer.config.musubi_config import MusubiConfigBuilder
from lora_trainer.engine.downloader import download_model_suite
from lora_trainer.engine.musubi_runner import run_musubi_pipeline
from lora_trainer.utils.colab_utils import auto_disconnect

res = [int(x.strip()) for x in Resolution.split(",")]
tf = [int(x.strip()) for x in Target_Frames.split(",")]

weights = download_model_suite(Model_Type, weights_dir="/content/models")

builder = MusubiConfigBuilder(
    model_name=Model_Type,
    output_dir=Output_Directory,
    output_name=LoRA_Name,
)

v_list = []
for vd in [d.strip() for d in Video_Folders.split(",") if d.strip()]:
    v_list.append({
        "path": vd,
        "repeats": Num_Repeats,
        "frame_extraction": Frame_Extraction,
        "target_frames": tf,
        "frame_stride": Frame_Stride,
        "frame_sample": Frame_Sample,
        "max_frames": Max_Frames,
    })

dataset_toml = "/content/dataset_video.toml"
builder.build_dataset_toml(
    dataset_path=dataset_toml,
    resolution=res,
    video_folders=v_list,
)

vae_path = weights.get("vae", "")
t5_path = weights.get("text_encoder1", "")
cv_path = weights.get("clip_vision", None)
dit_path = weights.get("dit", "")

cache_latents = builder.build_cache_latents_args(dataset_toml, vae_path, cv_path)
cache_te = builder.build_cache_text_encoder_args(dataset_toml, t5_path)

sample_txt = "/content/prompt_video.txt"
with open(sample_txt, "w", encoding="utf-8") as f:
    f.write(f"{Sample_Prompt} --w {res[0]} --h {res[1]} --f {tf[0]}\n")

train_cmd = builder.build_train_args(
    dataset_config_path=dataset_toml,
    dit_model_path=dit_path,
    learning_rate=Learning_Rate,
    network_dim=32,
    network_alpha=16,
    max_train_epochs=Max_Train_Epochs,
    save_every_n_epochs=Save_Every_N_Epochs,
    timestep_sampling=Timestep_Sampling,
    timestep_boundary=Timestep_Boundary if "wan22" in Model_Type.lower() else None,
    sample_prompt_file=sample_txt if Sample_Every_N_Steps > 0 else None,
    sample_every_n_steps=Sample_Every_N_Steps,
)

run_musubi_pipeline(
    musubi_dir="/content/musubi-tuner",
    cache_latents_cmd=cache_latents,
    cache_text_encoder_cmd=cache_te,
    train_cmd=train_cmd,
)

if Auto_Disconnect:
    auto_disconnect(delay_seconds=120, enabled=True)